In [1]:
# ============================================================
# CELL 1 — Setup & Install
# ============================================================
# الهدف: الانتقال للمجلد الآمن وتثبيت جميع المكتبات المطلوبة
#   - qdrant-client : قاعدة بيانات المتجهات (Vector DB)
#   - transformers  : تحميل نماذج DINOv2 (Image Embedding) و Qwen (LLM)
#   - accelerate    : مطلوب لتشغيل النماذج على GPU بكفاءة
#   - gradio        : واجهة تفاعلية لتجربة الـ RAG
#   - Pillow        : فتح ومعالجة الصور قبل إدخالها للنموذج
# ============================================================

%cd /content
!pip install qdrant-client transformers accelerate gradio Pillow -q


/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 6.2 MB/s eta 0:00:00


In [2]:
# ============================================================
# CELL 2 — Mount Google Drive & Load Pre-computed Embeddings
# ============================================================
# الهدف: تحميل الـ embeddings الجاهزة (image_vector + text_vector)
#   من ملف pickle المخزّن على Google Drive.
#   هذه النقاط تمثل مجموعة EGMM الأثرية متعددة الوسائط.
# ============================================================


import pickle
from google.colab import drive

print("Connecting to Google Drive...")
drive.mount('/content/drive', force_remount=True)


Connecting to Google Drive...
Mounted at /content/drive


In [3]:

# ── المسار المباشر لملف الـ Embeddings ──
PICKLE_PATH = "/content/drive/MyDrive/EgMM-Corpus/egmm_multimodal_embeddings.pkl"

print("Loading embeddings from pickle file...")
with open(PICKLE_PATH, "rb") as f:
    qdrant_points = pickle.load(f)

print(f"Done! Total points loaded: {len(qdrant_points)}")
# كل نقطة تحتوي على: id, image_vector (768d), text_vector (768d), payload


Loading embeddings from pickle file...
Done! Total points loaded: 4654


In [4]:
# ============================================================
# CELL 3 — Build Qdrant In-Memory Vector Database
# ============================================================
# الهدف: إنشاء قاعدة بيانات Qdrant في الـ RAM وتحميل
#   جميع النقاط فيها بـ named vectors:
#     - "image_vector" : من DINOv2 (768 بُعد، Cosine similarity)
#     - "text_vector"  : من BGE   (768 بُعد، Cosine similarity)
#   Named vectors تتيح البحث بكل نوع على حدة.
# ============================================================

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

qdrant_client = QdrantClient(":memory:")
COLLECTION_NAME = "egmm_corpus_multimodal"

print(f"Creating collection: '{COLLECTION_NAME}'...")
qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "image_vector": VectorParams(size=768, distance=Distance.COSINE),
        "text_vector":  VectorParams(size=768, distance=Distance.COSINE),
    }
)

# ── تحويل النقاط إلى PointStruct ──
points_to_upload = [
    PointStruct(
        id=p["id"],
        vector={
            "image_vector": p["image_vector"],
            "text_vector":  p["text_vector"]
        },
        payload=p["payload"]   # Concept_Name, Text_Chunk, Full_Description, …
    )
    for p in qdrant_points
]

print(f"Uploading {len(points_to_upload)} points...")
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points_to_upload)
print("Qdrant DB is ready!")


Creating collection: 'egmm_corpus_multimodal'...
Uploading 4654 points...
Qdrant DB is ready!


In [5]:
# ============================================================
# CELL 4 — Load AI Models (DINOv2 + Qwen LLM)
# ============================================================
# نموذجان يتم تحميلهما على GPU:
#
#   1. DINOv2-base (facebook/dinov2-base)
#      ─ نموذج Vision Transformer يحوّل أي صورة إلى متجه 768 بُعد.
#      ─ يُستخدم لتحويل الصورة التي يرفعها المستخدم إلى embedding
#        ثم يُقارَن بـ image_vectors في Qdrant (RAG بالصورة).
#
#   2. Qwen1.5-0.5B-Chat
#      ─ نموذج LLM خفيف يعمل على T4 GPU بدون مشاكل ذاكرة.
#      ─ يستقبل الـ Context المسترجع من Qdrant ويولّد
#        الرد النهائي (وصف تاريخي أو قصة شخصية).
# ============================================================

import torch
from transformers import AutoImageProcessor, AutoModel
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── 4a: DINOv2 — Image Embedding Model ──
print("\nLoading DINOv2 image encoder...")
DINO_MODEL_NAME = "facebook/dinov2-base"
img_processor = AutoImageProcessor.from_pretrained(DINO_MODEL_NAME)
img_model = AutoModel.from_pretrained(DINO_MODEL_NAME).to(device)
img_model.eval()
print("DINOv2 ready  (output dim: 768)")

# ── 4b: Qwen — Local Chat LLM ──
print("\nLoading Qwen1.5-0.5B-Chat LLM...")
LLM_MODEL_NAME = "Qwen/Qwen1.5-0.5B-Chat"
tokenizer    = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
local_model  = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME, torch_dtype="auto", device_map="auto"
)
print("Qwen LLM ready!")


Using device: cpu

Loading DINOv2 image encoder...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

Qwen LLM ready!


In [6]:
# ============================================================
# CELL 5 — Image → Embedding Helper
# ============================================================
# الدالة الأساسية لتحويل صورة إلى متجه (image embedding):
#   1. تفتح الصورة وتحوّلها إلى RGB.
#   2. تمررها عبر DINOv2 وتستخرج CLS token (أول token).
#   3. ترجع قائمة (list) من 768 قيمة جاهزة للبحث في Qdrant.
#
# هذه الدالة هي "الجسر" بين صورة المستخدم وقاعدة البيانات.
# ============================================================

from PIL import Image
import torch
import numpy as np

def get_image_embedding(image_input) -> list:
    """
    تحويل صورة إلى embedding vector (768d) باستخدام DINOv2.

    المدخل: PIL.Image أو مسار ملف (str)
    المخرج: list[float] بطول 768
    """
    # ── فتح الصورة إذا كانت مساراً ──
    if isinstance(image_input, str):
        image_input = Image.open(image_input)
    image_rgb = image_input.convert("RGB")

    # ── المعالجة وإدخالها للنموذج ──
    inputs = img_processor(images=image_rgb, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = img_model(**inputs)

    # ── استخراج CLS token كممثل للصورة كاملة ──
    cls_vector = outputs.last_hidden_state[:, 0, :]        # shape: (1, 768)
    return cls_vector.cpu().numpy().flatten().tolist()     # list of 768 floats


print("get_image_embedding() is ready!")


get_image_embedding() is ready!


In [7]:
# ============================================================
# CELL 6 — Core RAG Functions
# ============================================================
# يحتوي على دالتين رئيسيتين:
#
#   get_strict_summary(query_vector, search_type, limit)
#     ─ تبحث في Qdrant بالمتجه المعطى.
#     ─ تجمع الـ Chunks وتولّد وصفاً أكاديمياً دقيقاً
#       بأسلوب مرشد سياحي خبير.
#
#   get_personalized_story(query_vector, search_type, user_persona, limit)
#     ─ نفس البحث في Qdrant، لكن تولّد قصة شخصية
#       مُكيَّفة حسب نوع المستخدم (طفل/بالغ × ذكر/أنثى).
#     ─ تتحكم في التنسيق: فقرتان قصيرتان بدون نقاط.
# ============================================================

from IPython.display import display, Markdown

# ─────────────────────────────────────────────────────────────
# دالة 1: الوصف التاريخي الدقيق (Tour Guide Mode)
# ─────────────────────────────────────────────────────────────
def get_strict_summary(query_vector: list, search_type: str = "image_vector", limit: int = 3):
    """
    ترجع (response_text, raw_context).
    search_type: "image_vector" أو "text_vector"
    """
    print(f"Searching Qdrant with [{search_type}]...")

    # ── البحث عن أقرب النتائج ──
    search_results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_vector,
        using=search_type,
        limit=limit
    ).points

    if not search_results:
        return "Welcome, traveler! No context found for this artifact.", ""

    # ── تجميع النصوص وإزالة التكرار ──
    raw_chunks = [
        r.payload.get("Text_Chunk", r.payload.get("Full_Description", ""))
        for r in search_results
    ]
    unique_chunks = list(dict.fromkeys(raw_chunks))     # حذف التكرار مع الحفاظ على الترتيب
    combined_context = "\n".join(f"Fact: {c}" for c in unique_chunks)

    # ── Prompt: وصف أكاديمي منظّم ──
    prompt = f"""You are an expert Egyptologist Tour Guide. Welcome tourists and provide a comprehensive, detailed explanation of this artifact using ONLY the facts below.

Context:
{combined_context}

Output Instructions:
1. Provide ALL details from the facts. Do NOT shorten or omit details.
2. Structure your response into clear, readable lines.
3. Use a new line for each distinct historical fact.
4. Do NOT merge everything into a single long block.
5. Write in English.

Tour Guide Response:"""

    # ── توليد الرد ──
    try:
        messages = [{"role": "user", "content": prompt}]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        model_inputs = tokenizer([text], return_tensors="pt").to(device)

        generated_ids = local_model.generate(
            model_inputs.input_ids,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=False
        )
        output_ids = [o[len(i):] for i, o in zip(model_inputs.input_ids, generated_ids)]
        response = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]
        return response.strip(), combined_context
    except Exception as e:
        return f"Generation failed: {e}", combined_context


# ─────────────────────────────────────────────────────────────
# دالة 2: القصة الشخصية (Storytelling Mode)
# ─────────────────────────────────────────────────────────────
# def get_personalized_story(
#     query_vector: list,
#     search_type: str  = "image_vector",
#     user_persona: str = "man_adult",
#     limit: int        = 3
# ):
#     """
#     ترجع (story_text, raw_context).
#     user_persona: "boy_child" | "girl_child" | "man_adult" | "woman_adult"
#     """
#     print(f"Searching Qdrant with [{search_type}] for persona [{user_persona}]...")

#     search_results = qdrant_client.query_points(
#         collection_name=COLLECTION_NAME,
#         query=query_vector,
#         using=search_type,
#         limit=limit
#     ).points

#     if not search_results:
#         return "Welcome, traveler! No context found for this artifact.", ""

#     # ── تجميع الـ Chunks بدون تكرار ──
#     raw_chunks = [
#         r.payload.get("Text_Chunk", r.payload.get("Full_Description", ""))
#         for r in search_results
#     ]
#     unique_chunks  = list(dict.fromkeys(raw_chunks))
#     combined_context = "\n".join(f"- {c}" for c in unique_chunks)

#     # ── تحديد أسلوب الكتابة حسب الشخصية ──
#     persona_map = {
#         "boy_child":    "Character: An Egyptian boy named Youssef. Style: An exciting adventure story about standing in front of this historic Cairo landmark, looking up at its grand stonework.",
#         "girl_child":   "Character: An Egyptian girl named Fatma. Style: A beautiful, warm fairytale about exploring the magical stone architecture of this Old Cairo site.",
#         "man_adult":    "Style: An epic historical chronicle for an adult man, focusing on the tactical strength, military architecture, and the Fatimid vizier Badr al-Jamali.",
#         "woman_adult":  "Style: An elegant, descriptive narrative for an adult woman, focusing on the art, cultural atmosphere, and rich history of Old Cairo."
#     }
#     persona_instructions = persona_map.get(user_persona, "Write a clear historical narrative.")

#     # ── Prompt: قصة في فقرتين منضبطتين ──
#     prompt = f"""You are a strict historical storyteller. Use ONLY the factual context provided. Stay completely in Cairo, Egypt. Do NOT invent fictional places.

# [Factual Context]:
# {combined_context}

# [Persona Style]:
# {persona_instructions}

# [Strict Length & Format Rules]:
# 1. Your story MUST be exactly 2 short paragraphs total.
# 2. Each paragraph must be 2-3 sentences only.
# 3. Separate the two paragraphs with a single blank line.
# 4. Do NOT use bullet points. Write as a flowing narrative.
# 5. Complete both paragraphs fully. Do NOT cut off mid-sentence. Write in English.

# Story Response:"""

#     try:
#         messages = [{"role": "user", "content": prompt}]
#         text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
#         model_inputs = tokenizer([text], return_tensors="pt").to(device)

#         generated_ids = local_model.generate(
#             model_inputs.input_ids,
#             max_new_tokens=400,
#             temperature=0.1,
#             do_sample=False
#         )
#         output_ids = [o[len(i):] for i, o in zip(model_inputs.input_ids, generated_ids)]
#         response = tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0]
#         return response.strip(), combined_context
#     except Exception as e:
#         return f"Story generation failed: {e}", combined_context


# print("RAG core functions (get_strict_summary + get_personalized_story) are ready!")


RAG core functions (get_strict_summary + get_personalized_story) are ready!


In [ ]:
# ============================================================
# CELL 7 — Full Image-Based RAG Pipeline
# ============================================================
# دالة شاملة تجمع كل المراحل في خطوة واحدة:
#   1. تستقبل صورة (PIL Image أو مسار).
#   2. تحوّلها إلى embedding عبر DINOv2 (Cell 5).
#   3. تبحث في Qdrant بـ image_vector.
#   4. تُزيل التكرار من النتائج.
#   5. تولّد الرد المطلوب (وصف أو قصة) حسب mode و persona.
#
# مفتاح الـ RAG بالصورة: الخطوات 2+3 (embedding → vector search)
# هي ما يجعل النظام يفهم الصورة بدون caption أو نص.
# ============================================================

def ankh_rag_pipeline(
    image_input,
    mode: str    = "description",
    persona: str = "man_adult"
) -> tuple:
    """
    المدخل:
        image_input : PIL.Image أو str (مسار الصورة)
        mode        : "description" → وصف تاريخي دقيق
                      "story"       → قصة شخصية مُكيَّفة
        persona     : "man_adult" | "woman_adult" | "boy_child" | "girl_child"

    المخرج:
        (response_text: str, retrieved_context: str)
    """
    print("=" * 55)
    print(f"Mode: {mode.upper()}  |  Persona: {persona}")
    print("=" * 55)

    # ── الخطوة 1: تحويل الصورة إلى embedding (768d) ──
    print("Step 1: Generating image embedding via DINOv2...")
    query_vector = get_image_embedding(image_input)
    print(f"        Embedding generated. Dim = {len(query_vector)}")

    # ── الخطوة 2: اختيار الدالة المناسبة ──
    if mode == "story":
        print("Step 2: Running Personalized Story RAG...")
        return get_personalized_story(
            query_vector=query_vector,
            search_type="image_vector",
            user_persona=persona
        )
    else:
        print("Step 2: Running Historical Description RAG...")
        return get_strict_summary(
            query_vector=query_vector,
            search_type="image_vector"
        )


print("ankh_rag_pipeline() is ready!")

# ── اختبار سريع بأول نقطة حقيقية من قاعدة البيانات ──
print("\n--- Quick sanity test (using stored vector, not a real image) ---")
test_vector = qdrant_points[0]["image_vector"]
test_response, test_context = get_strict_summary(test_vector)
print("\nRetrieved Context:")
print(test_context[:300], "...")
print("\nGenerated Response:")
display(Markdown(test_response))


In [ ]:
# ============================================================
# CELL 8 — Gradio Interactive Interface
# ============================================================
# واجهة تفاعلية كاملة تتيح للمستخدم:
#   1. رفع صورة أثرية.
#   2. اختيار نوع الاستجابة: وصف تاريخي أو قصة شخصية.
#   3. اختيار الشخصية (للقصة فقط).
#   4. رؤية:
#      ─ الرد النهائي (وصف/قصة).
#      ─ الـ Context المسترجع من Qdrant (للـ Debugging).
#      ─ بُعد الـ Embedding الناتج عن الصورة.
# ============================================================

import gradio as gr
from PIL import Image

def gradio_handler(image, mode, persona):
    """
    الدالة الرئيسية المرتبطة بواجهة Gradio.
    تستدعي ankh_rag_pipeline وتُعيد النتائج.
    """
    if image is None:
        return "Please upload an image first.", "No image provided.", "—"

    try:
        # ── تشغيل الـ RAG Pipeline ──
        response, context = ankh_rag_pipeline(
            image_input=image,
            mode=mode,
            persona=persona
        )
        embedding_info = f"DINOv2 image_vector dim: 768 | Searched in: {len(qdrant_points)} DB points"
        return response, context, embedding_info

    except Exception as e:
        err = f"Error: {str(e)}"
        return err, err, "—"


# ── تعريف واجهة Gradio ──
with gr.Blocks(title="Ankh RAG — Egyptian Heritage Guide", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🏛️ Ankh RAG — Egyptian Heritage Tour Guide
    Upload an image of an Egyptian artifact or historic site.
    The system will **embed the image → search the vector database → generate a response**.
    """)

    with gr.Row():
        # ── العمود الأيسر: المدخلات ──
        with gr.Column(scale=1):
            img_input = gr.Image(
                type="pil",
                label="Upload Artifact / Site Image"
            )
            mode_radio = gr.Radio(
                choices=["description", "story"],
                value="description",
                label="Response Mode",
                info='"description" = academic tour guide | "story" = personalized narrative'
            )
            persona_dropdown = gr.Dropdown(
                choices=["man_adult", "woman_adult", "boy_child", "girl_child"],
                value="man_adult",
                label="Visitor Persona",
                info="Only used in Story mode"
            )
            submit_btn = gr.Button("Generate Response 🔍", variant="primary")

        # ── العمود الأيمن: المخرجات ──
        with gr.Column(scale=1):
            output_response = gr.Textbox(
                label="Generated Response",
                lines=12,
                placeholder="Response will appear here..."
            )
            output_context = gr.Textbox(
                label="Retrieved Context from Qdrant (RAG Source)",
                lines=6,
                placeholder="Retrieved text chunks will appear here..."
            )
            output_info = gr.Textbox(
                label="Embedding Info",
                lines=1,
                interactive=False
            )

    # ── ربط الزر بالدالة ──
    submit_btn.click(
        fn=gradio_handler,
        inputs=[img_input, mode_radio, persona_dropdown],
        outputs=[output_response, output_context, output_info]
    )

    gr.Markdown("""
    ---
    **How it works:**
    1. Your image is encoded to a **768-dim vector** using DINOv2.
    2. The vector is compared against all entries in the **Qdrant in-memory DB** (Cosine similarity).
    3. The top-3 closest text chunks are retrieved.
    4. Qwen LLM generates the final response from those chunks.
    """)

print("Launching Gradio interface...")
demo.launch(debug=True, share=True)
